# Q-Learning for Games: Teaching an Agent Tic-Tac-Toe Through Self-Play

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/tic_tac_toe_q_learning.ipynb)

Two Q-learning agents teach each other tic-tac-toe through self-play. No strategy is programmed — the agents discover forks, blocking, and centre-first openings purely from the reward signal.

**Blog post:** [sesen.ai/blog/q-learning-games-tic-tac-toe-self-play](https://sesen.ai/blog/q-learning-games-tic-tac-toe-self-play)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict

## 1. The Environment

In [ ]:
class TicTacToe:
    """Tic-tac-toe environment. Board is a flat array of 9 cells.
    Values: 0=empty, 1=X, -1=O."""

    def __init__(self):
        self.state = np.zeros(9, dtype=int)

    def reset(self):
        self.state = np.zeros(9, dtype=int)
        return self.state.copy()

    def available_actions(self):
        return np.where(self.state == 0)[0]

    def step(self, action, marker):
        self.state[action] = marker
        if self._is_winner():
            return self.state.copy(), 1, True, 'win'
        elif len(self.available_actions()) == 0:
            return self.state.copy(), 0, True, 'draw'
        return self.state.copy(), 0, False, 'ongoing'

    def _is_winner(self):
        b = self.state.reshape(3, 3)
        for i in range(3):
            if abs(b[i].sum()) == 3: return True
            if abs(b[:, i].sum()) == 3: return True
        if abs(np.diag(b).sum()) == 3: return True
        if abs(np.diag(np.fliplr(b)).sum()) == 3: return True
        return False

    def render(self):
        symbols = {1: 'X', -1: 'O', 0: '.'}
        b = self.state.reshape(3, 3)
        for row in b:
            print(' '.join(symbols[c] for c in row))
        print()

## 2. The Q-Learning Agent

In [ ]:
class QLearningAgent:
    def __init__(self, marker, epsilon=1.0, lr=1.0,
                 gamma=0.95, final_epsilon=0.05):
        self.marker = marker       # 1 for X, -1 for O
        self.epsilon = epsilon
        self.lr = lr
        self.gamma = gamma
        self.final_epsilon = final_epsilon
        self.q_table = {}          # {tuple(state): np.array(9)}

    def _get_q(self, state):
        key = tuple(state)
        if key not in self.q_table:
            q = np.full(9, np.nan)
            q[state == 0] = 0.0    # only empty cells get Q-values
            self.q_table[key] = q
        return self.q_table[key]

    def pick_action(self, state):
        available = np.where(state == 0)[0]
        if np.random.rand() < self.epsilon:
            return np.random.choice(available)
        q = self._get_q(state)
        available_q = [(a, q[a]) for a in available]
        max_q = max(v for _, v in available_q)
        best = [a for a, v in available_q if v == max_q]
        return random.choice(best)

    def update(self, state, action, reward, next_state, done):
        q = self._get_q(state)
        if done:
            target = reward
        else:
            next_q = self._get_q(next_state)
            target = reward + self.gamma * np.nanmax(next_q)
        q[action] += self.lr * (target - q[action])

## 3. Self-Play Training

In [ ]:
np.random.seed(42)
random.seed(42)

env = TicTacToe()
agent_x = QLearningAgent(marker=1, epsilon=1.0, lr=1.0, gamma=0.95)
agent_o = QLearningAgent(marker=-1, epsilon=1.0, lr=1.0, gamma=0.95)
eps_decay = 2.5e-5
n_episodes = 100_000

# Track outcomes
outcomes = []  # 'x', 'o', or 'draw'

for ep in range(n_episodes):
    state = env.reset()
    agents = [agent_x, agent_o]
    if random.random() < 0.5:
        agents = [agent_o, agent_x]
    turn = 0
    history = []
    done = False

    while not done:
        agent = agents[turn % 2]
        s = state.copy()
        action = agent.pick_action(s)
        next_state, reward, done, info = env.step(action, agent.marker)
        history.append((agent, s, action, reward, next_state, done))

        if done:
            agent.update(s, action, reward, next_state, done)
            if info == 'win' and len(history) >= 2:
                other = agents[(turn + 1) % 2]
                prev = history[-2]
                other.update(prev[1], prev[2], -reward, next_state, True)
        else:
            agent.update(s, action, reward, next_state, done)

        state = next_state
        turn += 1

    # Record outcome
    if info == 'win':
        winner_marker = agent.marker
        outcomes.append('x' if winner_marker == 1 else 'o')
    else:
        outcomes.append('draw')

    for a in [agent_x, agent_o]:
        if a.epsilon > a.final_epsilon:
            a.epsilon -= eps_decay

print(f"Training complete. Q-table sizes: X={len(agent_x.q_table)}, O={len(agent_o.q_table)}")

## 4. Training Curves

In [ ]:
window = 1000
x_wins = np.array([1 if o == 'x' else 0 for o in outcomes])
o_wins = np.array([1 if o == 'o' else 0 for o in outcomes])
draws = np.array([1 if o == 'draw' else 0 for o in outcomes])

x_rate = np.convolve(x_wins, np.ones(window)/window, mode='valid')
o_rate = np.convolve(o_wins, np.ones(window)/window, mode='valid')
d_rate = np.convolve(draws, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 5))
episodes = range(window - 1, n_episodes)
ax.plot(episodes, x_rate, label='X wins', alpha=0.8)
ax.plot(episodes, o_rate, label='O wins', alpha=0.8)
ax.plot(episodes, d_rate, label='Draws', alpha=0.8, color='green')
ax.set_xlabel('Episode')
ax.set_ylabel(f'Rate ({window}-game rolling avg)')
ax.set_title('Self-Play Training: Win/Draw Rates Over Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation vs Random Opponent

In [ ]:
class RandomAgent:
    def __init__(self, marker):
        self.marker = marker
    def pick_action(self, state):
        return np.random.choice(np.where(state == 0)[0])

def evaluate(agent, opponent, n_games=5000, agent_goes_first=True):
    wins, draws, losses = 0, 0, 0
    env_eval = TicTacToe()
    for _ in range(n_games):
        state = env_eval.reset()
        players = [agent, opponent] if agent_goes_first else [opponent, agent]
        turn = 0
        done = False
        while not done:
            player = players[turn % 2]
            action = player.pick_action(state)
            state, reward, done, info = env_eval.step(action, player.marker)
            turn += 1
        if info == 'draw':
            draws += 1
        elif (turn - 1) % 2 == 0:  # last player (who won) was first
            if agent_goes_first:
                wins += 1
            else:
                losses += 1
        else:
            if agent_goes_first:
                losses += 1
            else:
                wins += 1
    return wins, draws, losses

# Save epsilon and set to 0 for greedy evaluation
old_eps_x, old_eps_o = agent_x.epsilon, agent_o.epsilon
agent_x.epsilon, agent_o.epsilon = 0, 0

random_o = RandomAgent(marker=-1)
random_x = RandomAgent(marker=1)
x_w, x_d, x_l = evaluate(agent_x, random_o, agent_goes_first=True)
o_w, o_d, o_l = evaluate(agent_o, random_x, agent_goes_first=False)

agent_x.epsilon, agent_o.epsilon = old_eps_x, old_eps_o

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, (w, d, l), title in [(ax1, (x_w, x_d, x_l), 'Agent X vs Random'),
                               (ax2, (o_w, o_d, o_l), 'Agent O vs Random')]:
    n = w + d + l
    bars = ax.bar(['Win', 'Draw', 'Loss'], [w/n*100, d/n*100, l/n*100],
                  color=['#2ecc71', '#3498db', '#e74c3c'])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'{bar.get_height():.0f}%', ha='center', fontweight='bold')
    ax.set_ylabel('Percentage')
    ax.set_title(title)
    ax.set_ylim(0, 100)
fig.suptitle('Trained Agents vs Random Opponent (5,000 games)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Inspecting the Q-Values

In [ ]:
def plot_q_board(ax, state, q_vals, title):
    """Plot a tic-tac-toe board coloured by Q-values."""
    symbols = {1: 'X', -1: 'O', 0: ''}
    valid = q_vals[~np.isnan(q_vals)]
    vmin, vmax = valid.min(), valid.max()
    absmax = max(abs(vmin), abs(vmax), 0.01)

    display = np.full((3, 3), np.nan)
    for i in range(3):
        for j in range(3):
            idx = i * 3 + j
            if state[idx] == 0:
                display[i, j] = q_vals[idx]

    im = ax.imshow(display, cmap='RdYlGn', vmin=-absmax, vmax=absmax, aspect='equal')

    for i in range(3):
        for j in range(3):
            idx = i * 3 + j
            s_val = state[idx]
            if s_val != 0:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                            facecolor='#e8e8e8', alpha=0.95, zorder=1))
                color = '#2980b9' if s_val == 1 else '#c0392b'
                ax.text(j, i, symbols[s_val], ha='center', va='center',
                       fontsize=28, fontweight='bold', color=color, zorder=2)
            else:
                v = q_vals[idx]
                if not np.isnan(v):
                    is_best = (v == np.nanmax(q_vals))
                    if is_best:
                        ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                    facecolor='none', edgecolor='gold',
                                    linewidth=3, zorder=3))
                    txt_color = 'white' if abs(v) > absmax * 0.55 else 'black'
                    ax.text(j, i, f'{v:+.2f}', ha='center', va='center',
                           fontsize=12, fontweight='bold', color=txt_color, zorder=2)

    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=12, fontweight='bold')
    for k in range(4):
        ax.axhline(y=k-0.5, color='#888', linewidth=1)
        ax.axvline(x=k-0.5, color='#888', linewidth=1)
    return im

# Three strategic situations
state_fork = np.array([1, -1, 0, 0, 1, 0, -1, 0, 0])
state_block = np.array([-1, 0, 0, -1, 1, 0, 0, 0, 1])
state_win = np.array([1, 1, 0, -1, -1, 0, 0, 0, 0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
situations = [
    (state_fork, agent_x._get_q(state_fork).copy(), 'Set Up a Fork'),
    (state_block, agent_x._get_q(state_block).copy(), 'Block or Lose'),
    (state_win, agent_x._get_q(state_win).copy(), 'Take the Win'),
]

for ax, (state, q_vals, title) in zip(axes, situations):
    im = plot_q_board(ax, state, q_vals, title)
    fig.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle("What the Agent Learned: Q-Values Reveal Its Strategy",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Play Against the Agent

In [ ]:
def play_against_agent(agent):
    """Play a game of tic-tac-toe against the trained agent."""
    env_play = TicTacToe()
    state = env_play.reset()
    agent.epsilon = 0  # greedy play

    print("You are O (-1). Agent is X (1).")
    print("Enter positions 0-8:")
    print("0 | 1 | 2")
    print("---------")
    print("3 | 4 | 5")
    print("---------")
    print("6 | 7 | 8")
    print()

    turn = 0
    done = False
    while not done:
        if turn % 2 == 0:  # Agent's turn (X)
            action = agent.pick_action(state)
            print(f"Agent plays position {action}")
        else:  # Human's turn (O)
            env_play.render()
            available = env_play.available_actions()
            while True:
                try:
                    action = int(input(f"Your move (available: {available}): "))
                    if action in available:
                        break
                    print("Invalid move!")
                except ValueError:
                    print("Enter a number 0-8")

        marker = 1 if turn % 2 == 0 else -1
        state, reward, done, info = env_play.step(action, marker)
        turn += 1

    env_play.render()
    if info == 'draw':
        print("It's a draw!")
    elif marker == 1:
        print("Agent wins!")
    else:
        print("You win!")

# Uncomment to play:
# play_against_agent(agent_x)

## Exercises

1. **Symmetry exploitation:** Tic-tac-toe has 8-fold symmetry (4 rotations x 2 reflections). Modify the agent to map all symmetric board states to a single canonical form. How much does this reduce the Q-table size? Does it speed up learning?

2. **SARSA comparison:** Implement SARSA (on-policy TD learning) by replacing `max_a Q(s', a')` with `Q(s', a'_actual)` where `a'_actual` is the action the agent actually takes. Compare convergence speed and final performance against Q-learning.

3. **Reward shaping:** Try different reward structures: small negative reward for draws (-0.1), or intermediate rewards for creating two-in-a-row. How does this affect the learned strategy?

4. **Teacher training:** Instead of self-play, train against a heuristic opponent that always blocks winning moves and takes winning moves when available. Compare the resulting agent's strength to the self-play agent.